# Bloque 2 — Demo: Comparativa de estrategias de embedding

Este notebook demuestra el experimento A/B/C/D/E presentado en el Bloque 2:
¿cuántos posts necesitamos embedear por usuario para obtener una representación fiel?

**Usamos IronMarch como dataset de referencia** porque:
- Ya tenemos todos los embeddings precomputados (evitamos esperar horas de GPU)
- Tiene ground truth verificable (Slavros = MOONLORD) que permite contextualizar los resultados
- Tamaño manejable: 836 usuarios activos, números que se pueden visualizar

### Las 5 estrategias

| Estrategia | Descripción | Ventaja | Limitación |
|---|---|---|---|
| **A** | Concatenar TODOS los posts → 1 embedding | Rápido, 1 llamada/usuario | Trunca a 50K chars — pierde contenido en usuarios prolíficos |
| **B** | 1 embedding por post → promedio (todos los posts) | Gold standard, nada se pierde | Computacionalmente inviable a escala |
| **C** | Top-50 posts más largos → promedio | Mucho más rápido que B | Pierde algo de fidelidad |
| **D** | Top-100 posts más largos → promedio | Mejor ratio calidad/coste | Punto óptimo identificado |
| **E** | Top-150 posts más largos → promedio | Mayor fidelidad que D | Rendimiento marginal decreciente |

> **Prerrequisito**: tener los archivos `.npz` de IronMarch precomputados en `IronMarch/results/` o en el directorio que indique el notebook de caso.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from scipy.spatial.distance import cdist
import plotly.graph_objects as go
import plotly.express as px

# Directorio donde están los .npz de IronMarch
IM_RESULTS = Path('../results/ironmarch')
print('Buscando resultados en:', IM_RESULTS.resolve())

**De dónde salen estos datos**: los `.npz` de `results/ironmarch/` (patrones `s5a_...` a `s5e_...`) y el `posts_clean.parquet` que se leen aquí vienen de una corrida real de `scripts/run_centroids_im_comparison.py` — este notebook no recalcula nada, solo lee resultados precomputados. Para reproducir esta comparativa A/C/D/E con tu propio dataset, ese script (y su equivalente para HackingForums/RaidForums, `scripts/run_centroids_hf_comparison.py`) es la plantilla. Las funciones subyacentes son `src.embeddings.compute_actor_centroids` (estrategias C/D/E, muestreo + promedio) y `src.embeddings.embed_users` (estrategia A, concatenación).

**Sobre estilometría computacional**: `src/stylometry.py` (`extract_features`, `compare_users`) es una implementación de referencia, pero calcula similitud coseno sobre *ratios de features estadísticas* — no es Burrows' Delta. El heatmap de Burrows' Delta que se ve en los casos prácticos se computó aparte; `src/stylometry.py` no es su fuente.

## 1. Cargar los embeddings precomputados

Cada archivo `.npz` contiene dos arrays:
- `user_ids`: los identificadores de los usuarios (strings)
- `vectors`: la matriz de embeddings, una fila por usuario, 4096 columnas (dimensiones del vector)

Un **embedding** de 4096 dimensiones es un punto en un espacio de 4096 dimensiones. No podemos visualizarlo directamente, pero podemos calcular distancias entre puntos: usuarios cercanos en ese espacio tienen estilos de escritura similares.

In [ ]:
def load_latest(pattern: str, base_dir: Path):
    """Carga el .npz más reciente que coincida con el patrón."""
    matches = sorted(base_dir.glob(pattern))
    if not matches:
        return None
    data = np.load(matches[-1], allow_pickle=True)
    return data['user_ids'].tolist(), data['vectors']

# Cargar cada estrategia
strats = {
    'A (concat)':   load_latest('s5a_embed_users_*.npz', IM_RESULTS),
    'B (full)':     load_latest('s5b_centroids_full_*.npz', IM_RESULTS),
    'C (top-50)':   load_latest('s5c_centroids_sampled_*.npz', IM_RESULTS),
    'D (top-100)':  load_latest('s5d_centroids_sampled100_*.npz', IM_RESULTS),
    'E (top-150)':  load_latest('s5e_centroids_sampled150_*.npz', IM_RESULTS),
}

for name, result in strats.items():
    if result is None:
        print(f'  {name}: ⚠️  no encontrado')
    else:
        ids, vecs = result
        print(f'  {name}: {len(ids):,} usuarios, dim={vecs.shape[1]}')

## 2. ¿Qué es el Spearman ρ y por qué lo usamos?

Queremos saber: si usamos solo 100 posts en lugar de todos, ¿obtenemos representaciones parecidas?

**No podemos comparar los vectores directamente** porque las estrategias no tienen exactamente los mismos usuarios. En cambio, comparamos el *ranking de similitudes*:

1. Para cada par de usuarios (A, B), calculamos su similitud coseno bajo la estrategia D
2. Hacemos lo mismo bajo la estrategia B (referencia)
3. Ordenamos todos los pares por similitud en cada estrategia
4. **Spearman ρ** mide si el orden es el mismo: ρ=1 significa rankings idénticos, ρ=0 es aleatorio

Si ρ(D vs B) = 0.860, significa que el 86% de la estructura de similitudes se preserva usando 100 posts en lugar de todos.

### ¿Por qué similitud coseno?

La similitud coseno mide el ángulo entre dos vectores, no la distancia euclidiana.
Dos usuarios con el mismo estilo de escritura tendrán vectores que apuntan en la misma dirección aunque tengan distinto volumen de texto. Rango: -1 (opuestos) a 1 (idénticos).

In [ ]:
def cosine_similarities_flat(ids, vecs):
    """Devuelve un dict {(uid_i, uid_j): similitud} para todos los pares."""
    # Normalizar vectores a longitud 1 para que el producto escalar = coseno
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    vecs_norm = vecs / np.where(norms == 0, 1, norms)
    sim_matrix = vecs_norm @ vecs_norm.T  # Matriz de similitudes NxN
    i_idx, j_idx = np.triu_indices(len(ids), k=1)  # Solo triángulo superior (sin diagonal)
    return {(ids[i], ids[j]): float(sim_matrix[i, j]) for i, j in zip(i_idx, j_idx)}

# Calcular similitudes para B (referencia) y para A, C, D, E
sim_cache = {}
for name, result in strats.items():
    if result is not None:
        ids, vecs = result
        sim_cache[name] = cosine_similarities_flat(ids, vecs)
        print(f'{name}: {len(sim_cache[name]):,} pares calculados')

In [ ]:
def spearman_vs_reference(ref_sims: dict, cmp_sims: dict, label: str):
    """Calcula Spearman ρ entre dos conjuntos de similitudes, solo para pares comunes."""
    common_pairs = set(ref_sims.keys()) & set(cmp_sims.keys())
    if len(common_pairs) < 10:
        print(f'  {label}: insuficientes pares comunes ({len(common_pairs)})')
        return None
    ref_vals = [ref_sims[p] for p in common_pairs]
    cmp_vals = [cmp_sims[p] for p in common_pairs]
    rho, pval = spearmanr(ref_vals, cmp_vals)
    print(f'  {label:<20} ρ = {rho:.4f}   (n = {len(common_pairs):,} pares, p < 0.001)')
    return rho

ref = 'B (full)'
if ref in sim_cache:
    print(f'Spearman ρ comparando cada estrategia vs. {ref} (gold standard):\n')
    results = {}
    for name in ['A (concat)', 'C (top-50)', 'D (top-100)', 'E (top-150)']:
        if name in sim_cache:
            results[name] = spearman_vs_reference(sim_cache[ref], sim_cache[name], name)
else:
    print('⚠️  Estrategia B no disponible — usando D como referencia')
    ref = 'D (top-100)'
    print(f'Spearman ρ comparando vs. {ref}:\n')
    results = {}
    for name in ['A (concat)', 'C (top-50)', 'E (top-150)']:
        if name in sim_cache:
            results[name] = spearman_vs_reference(sim_cache[ref], sim_cache[name], name)

## 3. Visualización — rendimiento marginal decreciente

El siguiente gráfico muestra cómo mejora el ρ al añadir más posts, y cuánto tiempo de GPU cuesta cada estrategia.

El **rendimiento marginal decreciente** es el principio económico clave aquí: cada post adicional aporta menos que el anterior. En algún punto, el coste de añadir más posts supera el beneficio en fidelidad.

In [ ]:
# Datos del experimento IronMarch (resultados reales precomputados)
summary = pd.DataFrame([
    {'Estrategia': 'A (concat)',   'Posts/usuario': '≤50K chars', 'Tiempo (min)': 12,  'ρ vs B': None,  'Nota': 'Trunca prolíficos'},
    {'Estrategia': 'B (full)',     'Posts/usuario': 'todos',       'Tiempo (min)': 540, 'ρ vs B': 1.000, 'Nota': 'Gold standard'},
    {'Estrategia': 'C (top-50)',   'Posts/usuario': 50,            'Tiempo (min)': 155, 'ρ vs B': 0.799, 'Nota': ''},
    {'Estrategia': 'D (top-100)',  'Posts/usuario': 100,           'Tiempo (min)': 213, 'ρ vs B': 0.860, 'Nota': '← óptimo'},
    {'Estrategia': 'E (top-150)',  'Posts/usuario': 150,           'Tiempo (min)': 252, 'ρ vs B': 0.896, 'Nota': ''},
])

# Si tenemos resultados calculados en esta sesión, actualizamos los valores
for name, rho in results.items():
    if rho is not None:
        summary.loc[summary['Estrategia'] == name, 'ρ vs B'] = round(rho, 4)

print(summary.to_string(index=False))

In [ ]:
# Gráfico: ρ vs tiempo — la curva de rendimiento decreciente
plot_data = summary[summary['ρ vs B'].notna() & (summary['Estrategia'] != 'B (full)')].copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_data['Tiempo (min)'],
    y=plot_data['ρ vs B'],
    mode='lines+markers+text',
    text=plot_data['Estrategia'],
    textposition='top center',
    marker=dict(size=12, color=['#888', '#E94E4E', '#4EE97A', '#888']),
    line=dict(color='#ccc', width=1)
))
fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='B (gold standard, 9h)', annotation_position='right')
fig.update_layout(
    title='Fidelidad vs. coste computacional — rendimiento marginal decreciente',
    xaxis_title='Tiempo de GPU (minutos)',
    yaxis_title='Spearman ρ vs. full centroid (B)',
    yaxis=dict(range=[0.7, 1.05]),
    height=450
)
fig.show()

## 4. ¿Cuándo usar cada estrategia?

La elección depende de la distribución de posts por usuario en tu dataset.
**Antes de elegir estrategia, siempre mira el percentil 90 de posts/usuario.**

In [ ]:
import sys
sys.path.insert(0, str(Path('..').resolve()))

try:
    posts_im = pd.read_parquet('../results/ironmarch/posts_clean.parquet')
    post_counts = posts_im.groupby('userid').size()
    p50 = post_counts.quantile(0.50)
    p90 = post_counts.quantile(0.90)
    p99 = post_counts.quantile(0.99)
    print(f'IronMarch — distribución de posts por usuario:')
    print(f'  Mediana (P50): {p50:.0f} posts')
    print(f'  P90:           {p90:.0f} posts')
    print(f'  P99:           {p99:.0f} posts')
    print()
    if p90 < 50:
        print('→ Con P90 < 50: C, D y E darán resultados casi idénticos. Usar C.')
    elif p90 < 100:
        print('→ Con P90 < 100: D añade algo sobre C pero no mucho. C o D.')
    else:
        print(f'→ Con P90 = {p90:.0f}: D (top-100) es el punto óptimo confirmado.')
except FileNotFoundError:
    print('posts_clean.parquet no disponible — ejecutar primero 01_ingenieria_datos.ipynb')
    print()
    print('Regla general:')
    print('  P90 < 50  → usar C (top-50)')
    print('  P90 < 100 → C o D')
    print('  P90 ≥ 100 → D (top-100) es el óptimo')
    print('  P90 ≥ 150 → E si el presupuesto de GPU lo permite')

## 5. Conclusión

**Para un foro de tamaño similar a IronMarch (836 usuarios activos, media ~190 posts/usuario):**

- **D (top-100 posts más largos)** es el punto óptimo: 86% de fidelidad con el 40% del tiempo de B
- El salto C→D (+6.1 puntos de ρ, +1h GPU) vale la pena
- El salto D→E (+3.6 puntos de ρ, +40min GPU) tiene rendimiento marginal decreciente
- **A (concat)** es útil como baseline rápido pero pierde información en usuarios prolíficos

**Aplicado a los casos prácticos:** usamos D directamente, sabiendo que está validado. Si un dataset tiene muy pocos posts por usuario (P90 < 50), bajamos a C.